In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csc_matrix
from sklearn.neighbors import NearestNeighbors

In [ ]:
valid_lines = []
with open('/content/movie_titles.xls', 'r', encoding='latin-1') as file:
    for line in file:
        try:
            fields = line.strip().split(',')
            if len(fields) == 3:
                valid_lines.append(fields)
        except Exception as e:
            pass

data = pd.DataFrame(valid_lines, columns=['ID do filme', 'Ano de lançamento', 'Título'])

In [ ]:
data.head(2)

,ID do filme,Ano de lançamento,Título
0,1,2003,Dinosaur Planet
1,2,2004,Isle of Man TT 2004 Review


In [ ]:
data_rating = pd.read_csv('combined_data_1.txt',header = None, names = ['Cust_Id', 'Rating'], usecols = [0,1])

In [ ]:
data_rating.head()

,Cust_Id,Rating
0,1:,NaN
1,1488844,3.0
2,822109,5.0
3,885013,4.0
4,30878,4.0


In [ ]:
data_rating.describe()

,Rating
count,713262.000000
mean,3.580231
std,1.089544
min,1.000000
25%,3.000000
50%,4.000000
75%,4.000000
max,5.000000


In [ ]:
data_rating.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 713452 entries, 0 to 713451
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   Cust_Id  713452 non-null  object 
 1   Rating   713262 non-null  float64
dtypes: float64(1), object(1)
memory usage: 10.9+ MB


In [ ]:
data_rating.isnull().sum()

Cust_Id      0
Rating     190
dtype: int64

In [ ]:
movie_start_rows = data_rating[data_rating['Cust_Id'].str.contains(':')]
movie_start_rows

,Cust_Id,Rating
0,1:,NaN
548,2:,NaN
694,3:,NaN
2707,4:,NaN
2850,5:,NaN
...,...,...
677449,185:,NaN
679747,186:,NaN
680039,187:,NaN
700089,188:,NaN


In [ ]:
movie_start_indices = movie_start_rows.index
movie_start_indices

Int64Index([     0,    548,    694,   2707,   2850,   3991,   5011,   5105,
             20016,  20112,
            ...
            670672, 673091, 676650, 676885, 677049, 677449, 679747, 680039,
            700089, 702639],
           dtype='int64', length=189)

In [ ]:
data_rating['ID do filme'] = ''

In [ ]:
for i in range(len(movie_start_indices) - 1):
    start_idx = movie_start_indices[i]
    end_idx = movie_start_indices[i + 1]

    movie_name = data_rating.loc[start_idx, 'Cust_Id']
    data_rating.loc[start_idx + 1:end_idx - 1, 'ID do filme'] = movie_name

In [ ]:
data_rating = data_rating[~data_rating['Cust_Id'].str.contains(':')]
data_rating

,Cust_Id,Rating,ID do filme
1,1488844,3.0,1:
2,822109,5.0,1:
3,885013,4.0,1:
4,30878,4.0,1:
5,823519,3.0,1:
...,...,...,...
713447,1624904,3.0,
713448,1183483,3.0,
713449,1893292,4.0,
713450,418339,4.0,


In [ ]:
data_rating.loc[:, 'Cust_Id'] = data_rating['Cust_Id'].astype(int)
data_rating

<ipython-input-20-84b3f0ed336a>:1: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  data_rating.loc[:, 'Cust_Id'] = data_rating['Cust_Id'].astype(int)


,Cust_Id,Rating,ID do filme
1,1488844,3.0,1:
2,822109,5.0,1:
3,885013,4.0,1:
4,30878,4.0,1:
5,823519,3.0,1:
...,...,...,...
713447,1624904,3.0,
713448,1183483,3.0,
713449,1893292,4.0,
713450,418339,4.0,


In [ ]:
data.head()

,ID do filme,Ano de lançamento,Título
0,1,2003,Dinosaur Planet
1,2,2004,Isle of Man TT 2004 Review
2,3,1997,Character
3,4,1994,Paula Abdul's Get Up & Dance
4,5,2004,The Rise and Fall of ECW


In [ ]:
data_rating['ID do filme'] = data_rating['ID do filme'].str.replace(':', '')
data_rating.head()

,Cust_Id,Rating,ID do filme
1,1488844,3.0,1
2,822109,5.0,1
3,885013,4.0,1
4,30878,4.0,1
5,823519,3.0,1


In [ ]:
df = data.merge(data_rating, how='inner', on='ID do filme')

df.head()

,ID do filme,Ano de lançamento,Título,Cust_Id,Rating
0,1,2003,Dinosaur Planet,1488844,3.0
1,1,2003,Dinosaur Planet,822109,5.0
2,1,2003,Dinosaur Planet,885013,4.0
3,1,2003,Dinosaur Planet,30878,4.0
4,1,2003,Dinosaur Planet,823519,3.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 702272 entries, 0 to 702271
Data columns (total 5 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   ID do filme        702272 non-null  object 
 1   Ano de lançamento  702272 non-null  object 
 2   Título             702272 non-null  object 
 3   Cust_Id            702272 non-null  int64  
 4   Rating             702272 non-null  float64
dtypes: float64(1), int64(1), object(3)
memory usage: 32.1+ MB


In [ ]:
df.isnull().sum()

ID do filme          0
Ano de lançamento    0
Título               0
Cust_Id              0
Rating               0
dtype: int64

In [ ]:
df.duplicated().sum()

0

In [ ]:
df.shape

(702272, 5)

In [ ]:
df.loc[df['Ano de lançamento'].isnull(), 'Título']

Series([], Name: Título, dtype: object)

In [ ]:
df = df.dropna(axis=0)
df.head()

,ID do filme,Ano de lançamento,Título,Cust_Id,Rating
0,1,2003,Dinosaur Planet,1488844,3.0
1,1,2003,Dinosaur Planet,822109,5.0
2,1,2003,Dinosaur Planet,885013,4.0
3,1,2003,Dinosaur Planet,30878,4.0
4,1,2003,Dinosaur Planet,823519,3.0


In [ ]:
df['ID do filme'] = pd.to_numeric(df['ID do filme'], errors='coerce').astype('Int64')
df['Cust_Id'] = pd.to_numeric(df['Cust_Id'], errors='coerce').astype('Int64')

In [ ]:
df['Ano de lançamento'].replace('NULL', np.nan, inplace=True)

df['Ano de lançamento'] = pd.to_numeric(df['Ano de lançamento'],
                                        errors='coerce').astype('Int64')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 702272 entries, 0 to 702271
Data columns (total 5 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   ID do filme        702272 non-null  Int64  
 1   Ano de lançamento  702272 non-null  Int64  
 2   Título             702272 non-null  object 
 3   Cust_Id            702272 non-null  Int64  
 4   Rating             702272 non-null  float64
dtypes: Int64(3), float64(1), object(1)
memory usage: 34.2+ MB


In [ ]:
df.columns

Index(['ID do filme', 'Ano de lançamento', 'Título', 'Cust_Id', 'Rating'], dtype='object')

In [ ]:
data = df.groupby(['ID do filme', 'Ano de lançamento', 'Título',
                   'Cust_Id'])['Rating'].median()
data = data.reset_index()
data.head()

,ID do filme,Ano de lançamento,Título,Cust_Id,Rating
0,1,2003,Dinosaur Planet,915,5.0
1,1,2003,Dinosaur Planet,2442,3.0
2,1,2003,Dinosaur Planet,3321,3.0
3,1,2003,Dinosaur Planet,4326,4.0
4,1,2003,Dinosaur Planet,11589,3.0


In [ ]:
matriz = data.pivot_table(values='Rating', index='Título', columns='Cust_Id')
matriz.fillna(0, inplace=True)

matriz_esparsa = csc_matrix(matriz)

In [ ]:
matriz

In [ ]:
model = NearestNeighbors(n_neighbors=5,
                         algorithm='auto',
                         metric="minkowski")
model.fit(matriz_esparsa)

NearestNeighbors()

In [ ]:
for posicao, titulo in enumerate(matriz.index):
    if 'Dr' in str(titulo):
        print(posicao, titulo)

49 Dragonheart
50 Drowning on Dry Land
166 The Lemon Drop Kid


In [ ]:
filme = matriz_esparsa[49, :].reshape(1, -1)
distancia, recomendacao = model.kneighbors(filme)

In [ ]:
recomendacao

array([[ 49, 133,  12,  68, 155]])

In [ ]:
for filme in range(len(recomendacao)):
    print(matriz.index[recomendacao[filme]])

Index(['Dragonheart', 'Screamers', 'Arachnid', 'Horror Vision',
       'The Bonesetter'],
      dtype='object', name='Título')
